In [ ]:
"""
01_eda.ipynb - Exploratory Data Analysis
Grade-4: Comprehensive visualization and data understanding
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ====================================
# 1. LOAD DATA (Banking77 Dataset)
# ====================================
print("📊 Loading Banking77 Dataset...")

# Download from Hugging Face datasets or use direct link
# For this example, we'll simulate loading - replace with actual download
# !pip install datasets
from datasets import load_dataset

dataset = load_dataset("banking77")
train_data = pd.DataFrame(dataset['train'])
test_data = pd.DataFrame(dataset['test'])

# Combine for full analysis
full_data = pd.concat([train_data, test_data], ignore_index=True)

print(f"✅ Dataset loaded successfully!")
print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Total samples: {len(full_data)}")

# ====================================
# 2. BASIC DATA EXPLORATION
# ====================================
print("\n" + "="*50)
print("📋 DATASET OVERVIEW")
print("="*50)

print("\nFirst 5 rows:")
print(full_data.head())

print("\nDataset Info:")
print(full_data.info())

print("\nStatistical Summary:")
print(full_data.describe())

print("\nMissing Values:")
print(full_data.isnull().sum())

print(f"\nNumber of Unique Intents: {full_data['label'].nunique()}")

# ====================================
# 3. INTENT DISTRIBUTION ANALYSIS
# ====================================
print("\n" + "="*50)
print("🎯 INTENT DISTRIBUTION")
print("="*50)

# Count samples per intent
intent_counts = full_data['label'].value_counts()

print(f"\nMost common intents:")
print(intent_counts.head(10))

print(f"\nLeast common intents:")
print(intent_counts.tail(10))

# Visualize intent distribution
plt.figure(figsize=(15, 6))
intent_counts.plot(kind='bar', color='steelblue')
plt.title('Distribution of Intents in Banking77 Dataset', fontsize=16, fontweight='bold')
plt.xlabel('Intent Label', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('../data/intent_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Check class balance
print(f"\nClass Balance Statistics:")
print(f"Mean samples per class: {intent_counts.mean():.2f}")
print(f"Std samples per class: {intent_counts.std():.2f}")
print(f"Min samples: {intent_counts.min()}")
print(f"Max samples: {intent_counts.max()}")

# ====================================
# 4. TEXT LENGTH ANALYSIS
# ====================================
print("\n" + "="*50)
print("📏 TEXT LENGTH ANALYSIS")
print("="*50)

# Calculate text lengths
full_data['text_length'] = full_data['text'].apply(len)
full_data['word_count'] = full_data['text'].apply(lambda x: len(x.split()))

print("\nText Length Statistics:")
print(full_data[['text_length', 'word_count']].describe())

# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(full_data['text_length'], bins=50, color='coral', edgecolor='black')
axes[0].set_title('Distribution of Text Length (Characters)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(full_data['word_count'], bins=50, color='lightgreen', edgecolor='black')
axes[1].set_title('Distribution of Word Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../data/text_length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ====================================
# 5. VOCABULARY ANALYSIS
# ====================================
print("\n" + "="*50)
print("📚 VOCABULARY ANALYSIS")
print("="*50)

# Tokenize and count words
all_words = ' '.join(full_data['text']).lower().split()
word_freq = Counter(all_words)

print(f"\nTotal words: {len(all_words)}")
print(f"Unique words: {len(word_freq)}")
print(f"\nTop 20 most common words:")
for word, count in word_freq.most_common(20):
    print(f"  {word}: {count}")

# Visualize top words
top_words = dict(word_freq.most_common(30))
plt.figure(figsize=(15, 6))
plt.bar(top_words.keys(), top_words.values(), color='purple', alpha=0.7)
plt.title('Top 30 Most Common Words', fontsize=16, fontweight='bold')
plt.xlabel('Words', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/top_words.png', dpi=300, bbox_inches='tight')
plt.show()

# ====================================
# 6. SAMPLE TEXTS PER INTENT
# ====================================
print("\n" + "="*50)
print("💬 SAMPLE TEXTS PER INTENT")
print("="*50)

# Show examples from different intents
for intent in full_data['label'].unique()[:5]:
    print(f"\nIntent {intent}:")
    samples = full_data[full_data['label'] == intent]['text'].head(3)
    for i, text in enumerate(samples, 1):
        print(f"  {i}. {text}")

# ====================================
# 7. SAVE PROCESSED DATA
# ====================================
print("\n" + "="*50)
print("💾 SAVING DATA")
print("="*50)

# Save full dataset
full_data.to_csv('../data/raw_data.csv', index=False)
train_data.to_csv('../data/train_raw.csv', index=False)
test_data.to_csv('../data/test_raw.csv', index=False)

print("✅ Data saved successfully!")
print(f"  - raw_data.csv: {len(full_data)} rows")
print(f"  - train_raw.csv: {len(train_data)} rows")
print(f"  - test_raw.csv: {len(test_data)} rows")

# ====================================
# 8. KEY INSIGHTS SUMMARY
# ====================================
print("\n" + "="*50)
print("🔍 KEY INSIGHTS")
print("="*50)

insights = f"""
Dataset: Banking77
Total Samples: {len(full_data)}
Number of Intents: {full_data['label'].nunique()}
Average Text Length: {full_data['text_length'].mean():.1f} characters
Average Word Count: {full_data['word_count'].mean():.1f} words
Vocabulary Size: {len(word_freq)} unique words

Class Balance: {'BALANCED' if intent_counts.std() < intent_counts.mean() * 0.2 else 'IMBALANCED'}

Most Common Intent: {intent_counts.index[0]} ({intent_counts.values[0]} samples)
Least Common Intent: {intent_counts.index[-1]} ({intent_counts.values[-1]} samples)
"""

print(insights)

with open('../data/eda_insights.txt', 'w') as f:
    f.write(insights)

print("\n✅ EDA Complete! Visualizations and insights saved.")